# Table 5 — Multi-view reliability of action-rule targeting

Stratified K-fold cross-validation for each article dataset.  Three complementary views of the same evaluation are reported (per Hastie, Tibshirani \& Friedman, *Elements of Statistical Learning*, Ch. 7, and Bates, Hastie \& Tibshirani 2021):

1. **In-sample (apparent).** Mine and score on the full dataset; optimistic upper bound by construction.
2. **Per-fold range.** Min / median / max across the five 80/20 train/test holdouts — captures fold-to-fold instability of rule discovery and OOF noise.
3. **CV aggregate.** Across-fold mean with the cluster-by-fold bootstrap 95\% CI (the inferential view recommended by Bates et al.).

Four targeting strategies are evaluated: `point`, `lower`, `lower_positive`, `risk_adjusted`.  Four metrics: `uplift_at_k`, `qini`, `auuc`, `profit_at_k`.

**Outputs:**
- `article/results/cv_multiview.csv` — new file consumed by `article/tables/table6_cv.py`; one row per `(dataset, strategy, metric, metric_target, view)`.
- `article/results/cv_results.csv` — preserved for backward compatibility (single-view CV aggregate; consumed by legacy downstream scripts).
- `article/results/cv_rule_records.csv` — preserved (per-rule per-fold records).

**Production parameters:** `n_splits=5`, `n_bootstrap=500`, `n_bootstrap_oof=1000`, `bootstrap_design='cluster_fold'`, `compute_insample_baseline=True`.

In [1]:
from __future__ import annotations
import sys, time, warnings
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / 'notebooks' / 'article') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'notebooks' / 'article'))

warnings.filterwarnings('ignore')
from _datasets import load_all
from action_rules import ActionRules

OUT_CSV = REPO_ROOT / 'article' / 'results' / 'cv_results.csv'
OUT_RULES = REPO_ROOT / 'article' / 'results' / 'cv_rule_records.csv'
OUT_MULTIVIEW = REPO_ROOT / 'article' / 'results' / 'cv_multiview.csv'
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

In [2]:
# Production-scale parameters (regenerate article-grade numbers).
# Smoke-scale defaults are commented next to each constant.
N_SPLITS = 5              # smoke: 3
N_BOOTSTRAP = 500         # smoke: 120
N_BOOTSTRAP_OOF = 1000    # smoke: 200
K_FRACTION = 0.20
SEED = 42
# 'cluster_fold' (default) bootstraps rule records within each fold and
# averages per-fold metrics across folds, so the CI estimates the same
# fold-mean quantity as the `mean` column.  Use 'oof_pool' only to
# reproduce the legacy pool-level statistic.
BOOTSTRAP_DESIGN = 'cluster_fold'

In [3]:
summary_frames = []
rule_frames = []
multiview_frames = []
for cfg in load_all():
    print(f'=== {cfg.name} ===')
    ar = ActionRules(
        min_stable_attributes=cfg.min_stable_attributes,
        min_flexible_attributes=cfg.min_flexible_attributes,
        min_undesired_support=cfg.min_undesired_support,
        min_desired_support=cfg.min_desired_support,
        min_undesired_confidence=cfg.min_undesired_confidence,
        min_desired_confidence=cfg.min_desired_confidence,
        intrinsic_utility_table=cfg.intrinsic_utility_table,
        transition_utility_table=cfg.transition_utility_table,
    )
    t0 = time.perf_counter()
    result = ar.cross_validate(
        cfg.df,
        stable_attributes=cfg.stable_attributes,
        flexible_attributes=cfg.flexible_attributes,
        target=cfg.target,
        target_undesired_state=cfg.target_undesired_state,
        target_desired_state=cfg.target_desired_state,
        n_splits=N_SPLITS,
        n_bootstrap=N_BOOTSTRAP,
        n_bootstrap_oof=N_BOOTSTRAP_OOF,
        bootstrap_design=BOOTSTRAP_DESIGN,
        k_fraction=K_FRACTION,
        random_state=SEED,
        use_sparse_matrix=cfg.use_sparse_matrix,
        compute_insample_baseline=True,
    )
    elapsed = time.perf_counter() - t0
    summary = result.strategy_summary.copy()
    summary.insert(0, 'dataset', cfg.name)
    summary_frames.append(summary)
    rules = result.rule_records.copy()
    rules.insert(0, 'dataset', cfg.name)
    rule_frames.append(rules)

    # ---- Build the three-view long-form rows for this dataset.
    # cv_aggregate: the same numbers as `summary` above, tagged with view name.
    cv_view = summary.copy()
    cv_view['view'] = 'cv_aggregate'
    cv_view = cv_view.rename(columns={})
    cv_view['range_min'] = pd.NA
    cv_view['range_max'] = pd.NA
    cv_view['range_median'] = pd.NA

    # fold_range: aggregate fold_summary -> min/max/median per (strategy, metric, target).
    fr = (
        result.fold_summary.groupby(['strategy', 'metric', 'metric_target'])['value']
        .agg(['min', 'max', 'median'])
        .reset_index()
        .rename(columns={'min': 'range_min', 'max': 'range_max', 'median': 'range_median'})
    )
    fr.insert(0, 'dataset', cfg.name)
    fr['view'] = 'fold_range'
    fr['mean'] = pd.NA
    fr['std'] = pd.NA
    fr['n_folds'] = result.n_splits
    fr['ci_lower'] = pd.NA
    fr['ci_upper'] = pd.NA

    # in_sample: scalar value per (strategy, metric, target).
    ins = result.insample_summary.copy() if result.insample_summary is not None else pd.DataFrame()
    if not ins.empty:
        ins = ins.rename(columns={'value': 'mean'})
        ins.insert(0, 'dataset', cfg.name)
        ins['view'] = 'in_sample'
        ins['std'] = pd.NA
        ins['n_folds'] = pd.NA
        ins['ci_lower'] = pd.NA
        ins['ci_upper'] = pd.NA
        ins['range_min'] = pd.NA
        ins['range_max'] = pd.NA
        ins['range_median'] = pd.NA

    multiview_frames.append(pd.concat([ins, fr, cv_view], ignore_index=True, sort=False))

    jaccard = result.rule_stability['jaccard'].mean() if result.rule_stability is not None else float('nan')
    print(f'  rules/fold = {result.n_rules_per_fold}; mean Jaccard overlap = {jaccard:.3f}; elapsed = {elapsed:.1f}s')

summary_df = pd.concat(summary_frames, ignore_index=True)
rule_df = pd.concat(rule_frames, ignore_index=True)
multiview_df = pd.concat(multiview_frames, ignore_index=True, sort=False)

# Canonical multi-view column order.
multiview_cols = [
    'dataset', 'strategy', 'metric', 'metric_target', 'view',
    'mean', 'std', 'n_folds', 'ci_lower', 'ci_upper',
    'range_min', 'range_max', 'range_median',
]
multiview_df = multiview_df.reindex(columns=multiview_cols)

summary_df.to_csv(OUT_CSV, index=False)
rule_df.to_csv(OUT_RULES, index=False)
multiview_df.to_csv(OUT_MULTIVIEW, index=False)
print(f'\nwrote {OUT_CSV.relative_to(REPO_ROOT)}')
print(f'wrote {OUT_RULES.relative_to(REPO_ROOT)}')
print(f'wrote {OUT_MULTIVIEW.relative_to(REPO_ROOT)}')

=== Telco Customer Churn ===


  rules/fold = [25, 59, 30, 18, 28]; mean Jaccard overlap = 0.464; elapsed = 21.3s
=== UCI Bank Marketing ===


  rules/fold = [48, 52, 16, 25, 31]; mean Jaccard overlap = 0.562; elapsed = 117.9s
=== IBM Employee Attrition ===


  rules/fold = [27, 31, 20, 34, 37]; mean Jaccard overlap = 0.426; elapsed = 22.4s
=== Taiwan Credit Card Default ===


  rules/fold = [20, 23, 20, 20, 20]; mean Jaccard overlap = 0.948; elapsed = 37.7s

wrote article\results\cv_results.csv
wrote article\results\cv_rule_records.csv
wrote article\results\cv_multiview.csv


In [4]:
# Tidy preview: uplift@20% per dataset × strategy
pivot = (
    summary_df[(summary_df['metric'] == 'uplift_at_k') & (summary_df['metric_target'] == 'uplift')]
    .pivot(index='dataset', columns='strategy', values='mean')
    .round(4)
)
print('Uplift@20% (mean across folds), by dataset × strategy:')
print(pivot.to_string())

Uplift@20% (mean across folds), by dataset × strategy:
strategy                     lower  lower_positive   point  risk_adjusted
dataset                                                                  
IBM Employee Attrition      0.0110          0.0110  0.0112         0.0109
Taiwan Credit Card Default  0.0156          0.0156  0.0158         0.0157
Telco Customer Churn        0.0373          0.0373  0.0374         0.0374
UCI Bank Marketing          0.1124          0.1124  0.1124         0.1124


## Interpretation

For each dataset, four strategies are evaluated on the held-out folds. The `point` strategy ranks rules by the train-fold uplift point estimate; `lower` ranks by the lower 95% CI bound (more conservative); `lower_positive` further filters to rules whose lower bound is positive; `risk_adjusted` uses `point - 1.96·SE`.

When the utility tables yield realistic gain rows (`metric_target == 'gain'`), the targeting metrics are repeated with realized gain as the outcome — directly answering: *which strategy maximises out-of-sample monetary value?*

The `ci_lower` / `ci_upper` columns are computed under the **cluster-fold bootstrap design** (resample rules within each fold, compute metric per fold, average across folds — preserves the within-fold rule dependence). The CI and the `mean` column therefore estimate the same fold-mean target quantity. The fold-level `mean ± 1.96·std/√K` interval has below-nominal coverage (Bates, Hastie & Tibshirani 2021) and is not reported as an inferential interval; `std` is included only as a stability indicator.